# Feature Engineering Pipeline
# Datathon 2026 Round 1 — AIO OIA
# Creates: train_features.parquet, test_features.parquet

## Cell 1: Setup

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
import sys, os, json

sys.path.insert(0, '..')
os.makedirs('../src/data/processed', exist_ok=True)
os.makedirs('../report/figures', exist_ok=True)

RAW = "../src/data/raw"
PROCESSED = "../src/data/processed"

print("Setup complete.")


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\Users\LOQ\AppData\Roaming\Python\Python312\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "C:\Users\LOQ\AppData\Roaming\Python\Python312\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "C:\Users\LOQ\AppData\Roaming\Python\Python312\site-packages\ipykernel\kernelapp.py", line 739, in start
    self.io_loop.sta

AttributeError: _ARRAY_API not found


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\Users\LOQ\AppData\Roaming\Python\Python312\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "C:\Users\LOQ\AppData\Roaming\Python\Python312\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "C:\Users\LOQ\AppData\Roaming\Python\Python312\site-packages\ipykernel\kernelapp.py", line 739, in start
    self.io_loop.sta

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.



Setup complete.


## Cell 2: Load Data

In [2]:
# Training data
sales = pd.read_csv(f"{RAW}/sales.csv", parse_dates=["Date"])
sales = sales.sort_values("Date").reset_index(drop=True)

# Test dates from sample_submission
submission_template = pd.read_csv(f"{RAW}/sample_submission.csv", parse_dates=["Date"])
test_dates = submission_template[["Date"]].copy()
test_dates = test_dates.sort_values("Date").reset_index(drop=True)

print(f"Train: {sales.shape} | {sales['Date'].min().date()} -> {sales['Date'].max().date()}")
print(f"Test:  {test_dates.shape} | {test_dates['Date'].min().date()} -> {test_dates['Date'].max().date()}")

Train: (3833, 3) | 2012-07-04 -> 2022-12-31
Test:  (548, 1) | 2023-01-01 -> 2024-07-01


## Cell 3: Step 1 — Temporal Features

In [3]:
def add_temporal_features(df, date_col="Date"):
    df = df.copy()
    dt = df[date_col].dt
    df["year"]            = dt.year
    df["month"]           = dt.month
    df["quarter"]         = dt.quarter
    df["day_of_month"]    = dt.day
    df["day_of_week"]     = dt.dayofweek        # 0=Monday
    df["day_of_year"]     = dt.dayofyear
    df["week_of_year"]    = dt.isocalendar().week.astype(int)
    df["is_weekend"]      = (dt.dayofweek >= 5).astype(int)
    df["is_month_start"]  = (dt.day <= 3).astype(int)
    df["is_month_end"]    = (dt.day >= 28).astype(int)
    df["is_quarter_start"] = (dt.month.isin([1, 4, 7, 10]) & (dt.day <= 3)).astype(int)
    df["is_quarter_end"]   = (dt.month.isin([3, 6, 9, 12]) & (dt.day >= 28)).astype(int)
    # Circular encoding
    df["month_sin"] = np.sin(2 * np.pi * dt.month / 12)
    df["month_cos"] = np.cos(2 * np.pi * dt.month / 12)
    df["dow_sin"]   = np.sin(2 * np.pi * dt.dayofweek / 7)
    df["dow_cos"]   = np.cos(2 * np.pi * dt.dayofweek / 7)
    df["doy_sin"]   = np.sin(2 * np.pi * dt.dayofyear / 365)
    df["doy_cos"]   = np.cos(2 * np.pi * dt.dayofyear / 365)
    return df

train_feat = add_temporal_features(sales)
test_feat  = add_temporal_features(test_dates)

temporal_cols = [c for c in train_feat.columns if c not in ["Date", "Revenue", "COGS"]]
print(f"Temporal features added ({len(temporal_cols)}):", temporal_cols)

Temporal features added (18): ['year', 'month', 'quarter', 'day_of_month', 'day_of_week', 'day_of_year', 'week_of_year', 'is_weekend', 'is_month_start', 'is_month_end', 'is_quarter_start', 'is_quarter_end', 'month_sin', 'month_cos', 'dow_sin', 'dow_cos', 'doy_sin', 'doy_cos']


## Cell 4: Step 2 — Lag Features (SAFE — no data leakage)

In [4]:
# Build lookup from training data only
rev_lookup  = sales.set_index("Date")["Revenue"]
cogs_lookup = sales.set_index("Date")["COGS"]

def safe_lag(df, lookup, lag_days):
    """Lookup value lag_days ago — safe for future forecast, no leakage."""
    lag_dates = df["Date"] - pd.Timedelta(days=lag_days)
    return lag_dates.map(lookup)

# Basic lag features (365 and 730 days back — fully available for test period)
for df in [train_feat, test_feat]:
    df["rev_lag_365"]  = safe_lag(df, rev_lookup,  365)
    df["cogs_lag_365"] = safe_lag(df, cogs_lookup, 365)
    df["rev_lag_730"]  = safe_lag(df, rev_lookup,  730)
    df["cogs_lag_730"] = safe_lag(df, cogs_lookup, 730)
    df["rev_lag_364"]  = safe_lag(df, rev_lookup,  364)   # 1-day offset
    df["rev_lag_366"]  = safe_lag(df, rev_lookup,  366)   # 1-day offset

# Fast rolling mean centered on lag_365 date (sorted-index lookup)
def rolling_around_lag_fast(df, lookup, lag=365, window=7):
    """Rolling mean of 'window' days centered on the date lag days before."""
    lookup_indexed = lookup.sort_index()
    all_idx = lookup_indexed.index
    results = []
    half = window // 2
    for date in df["Date"]:
        start = date - pd.Timedelta(days=lag + half)
        end   = date - pd.Timedelta(days=lag - half)
        mask  = (all_idx >= start) & (all_idx <= end)
        vals  = lookup_indexed[mask]
        results.append(vals.mean() if len(vals) > 0 else np.nan)
    return results

print("Computing rev_rolling7_lag365 (train)...")
train_feat["rev_rolling7_lag365"]  = rolling_around_lag_fast(train_feat, rev_lookup, window=7)
print("Computing rev_rolling7_lag365 (test)...")
test_feat["rev_rolling7_lag365"]   = rolling_around_lag_fast(test_feat,  rev_lookup, window=7)

print("Computing rev_rolling30_lag365 (train)...")
train_feat["rev_rolling30_lag365"] = rolling_around_lag_fast(train_feat, rev_lookup, window=30)
print("Computing rev_rolling30_lag365 (test)...")
test_feat["rev_rolling30_lag365"]  = rolling_around_lag_fast(test_feat,  rev_lookup, window=30)

# YoY ratio: Revenue / rev_lag_365 (captures growth factor)
train_feat["yoy_ratio"] = np.where(
    train_feat["rev_lag_365"] > 0,
    train_feat["Revenue"] / train_feat["rev_lag_365"],
    np.nan
)
# For test: use median YoY ratio from last 2 years of training
avg_yoy = train_feat[train_feat["year"] >= 2021]["yoy_ratio"].median()
test_feat["yoy_ratio"] = avg_yoy
print(f"\nAverage YoY ratio (2021-2022): {avg_yoy:.4f}")

# NaN report
for col in ["rev_lag_365", "rev_lag_730", "rev_rolling7_lag365"]:
    nan_train = train_feat[col].isna().sum()
    nan_test  = test_feat[col].isna().sum()
    print(f"{col}: train NaN={nan_train}, test NaN={nan_test}")

Computing rev_rolling7_lag365 (train)...
Computing rev_rolling7_lag365 (test)...
Computing rev_rolling30_lag365 (train)...
Computing rev_rolling30_lag365 (test)...

Average YoY ratio (2021-2022): 1.0643
rev_lag_365: train NaN=365, test NaN=183
rev_lag_730: train NaN=730, test NaN=0
rev_rolling7_lag365: train NaN=362, test NaN=180


## Cell 5: Step 3 — Holiday Features (Extended to 2024)

In [5]:
# All Vietnam public holidays 2012-2024
ALL_HOLIDAYS = {
    # New Year (fixed: Jan 1)
    **{f"{y}-01-01": "New_Year" for y in range(2012, 2025)},
    # Fixed national holidays
    **{f"{y}-04-30": "Reunification_Day" for y in range(2012, 2025)},
    **{f"{y}-05-01": "Labor_Day"         for y in range(2012, 2025)},
    **{f"{y}-09-02": "National_Day"      for y in range(2012, 2025)},
    **{f"{y}-03-08": "Women_Day"         for y in range(2012, 2025)},
    # Lunar New Year (Tet) — varies each year
    "2012-01-23": "Tet", "2012-01-24": "Tet", "2012-01-25": "Tet",
    "2013-02-10": "Tet", "2013-02-11": "Tet", "2013-02-12": "Tet",
    "2014-01-31": "Tet", "2014-02-01": "Tet", "2014-02-02": "Tet",
    "2015-02-19": "Tet", "2015-02-20": "Tet", "2015-02-21": "Tet",
    "2016-02-08": "Tet", "2016-02-09": "Tet", "2016-02-10": "Tet",
    "2017-01-28": "Tet", "2017-01-29": "Tet", "2017-01-30": "Tet",
    "2018-02-16": "Tet", "2018-02-17": "Tet", "2018-02-18": "Tet",
    "2019-02-05": "Tet", "2019-02-06": "Tet", "2019-02-07": "Tet",
    "2020-01-25": "Tet", "2020-01-26": "Tet", "2020-01-27": "Tet",
    "2021-02-12": "Tet", "2021-02-13": "Tet", "2021-02-14": "Tet",
    "2022-02-01": "Tet", "2022-02-02": "Tet", "2022-02-03": "Tet",
    "2023-01-22": "Tet", "2023-01-23": "Tet", "2023-01-24": "Tet",  # Tet Quy Mao
    "2024-02-10": "Tet", "2024-02-11": "Tet", "2024-02-12": "Tet",  # Tet Giap Thin
}

holiday_dates_series = pd.to_datetime(list(ALL_HOLIDAYS.keys()))

def add_holiday_features(df, holidays_dict, holiday_dates):
    df = df.copy()
    date_str = df["Date"].dt.strftime("%Y-%m-%d")
    df["is_holiday"]      = date_str.isin(holidays_dict.keys()).astype(int)
    df["is_tet"]          = date_str.map(holidays_dict).eq("Tet").astype(int)
    df["is_national_day"] = date_str.map(holidays_dict).isin(
        ["National_Day", "Reunification_Day", "Labor_Day"]
    ).astype(int)

    # Days to next holiday
    def days_to_next(d):
        future = holiday_dates[holiday_dates > d]
        return int((future.min() - d).days) if len(future) > 0 else 365

    # Days since last holiday
    def days_since_last(d):
        past = holiday_dates[holiday_dates <= d]
        return int((d - past.max()).days) if len(past) > 0 else 365

    df["days_to_holiday"]   = df["Date"].apply(days_to_next)
    df["days_since_holiday"] = df["Date"].apply(days_since_last)
    df["is_near_holiday"]   = (df["days_to_holiday"] <= 7).astype(int)
    df["is_post_holiday"]   = (df["days_since_holiday"] <= 3).astype(int)
    return df

train_feat = add_holiday_features(train_feat, ALL_HOLIDAYS, holiday_dates_series)
test_feat  = add_holiday_features(test_feat,  ALL_HOLIDAYS, holiday_dates_series)

print("Holiday features added.")
print(f"  Holidays in train: {train_feat['is_holiday'].sum()}")
print(f"  Tet days in train: {train_feat['is_tet'].sum()}")
print(f"  Holidays in test:  {test_feat['is_holiday'].sum()}")
print(f"  Tet days in test:  {test_feat['is_tet'].sum()}")

Holiday features added.
  Holidays in train: 81
  Tet days in train: 30
  Holidays in test:  15
  Tet days in test:  6


## Cell 6: Step 4 — Promotion Features

In [6]:
promos = pd.read_csv(f"{RAW}/promotions.csv", parse_dates=["start_date", "end_date"])

# Build daily promo features for ALL dates (train + test)
all_dates = pd.concat([train_feat[["Date"]], test_feat[["Date"]]]).drop_duplicates().sort_values("Date")

promo_rows = []
for _, row in all_dates.iterrows():
    d = row["Date"]
    active = promos[(promos["start_date"] <= d) & (promos["end_date"] >= d)]
    promo_rows.append({
        "Date":         d,
        "promo_count":  len(active),
        "avg_discount": active["discount_value"].mean() if len(active) > 0 else 0.0,
        "max_discount": active["discount_value"].max()  if len(active) > 0 else 0.0,
        "has_promo":    int(len(active) > 0),
    })

promo_daily = pd.DataFrame(promo_rows)
train_feat = train_feat.merge(promo_daily, on="Date", how="left")
test_feat  = test_feat.merge(promo_daily,  on="Date", how="left")

# Fill any remaining NaN with 0
for col in ["promo_count", "avg_discount", "max_discount", "has_promo"]:
    train_feat[col] = train_feat[col].fillna(0)
    test_feat[col]  = test_feat[col].fillna(0)

print(f"Promo features added.")
print(f"  Train active promo days: {(train_feat['has_promo'] == 1).sum()}")
print(f"  Test  active promo days: {(test_feat['has_promo'] == 1).sum()}")
print(f"  Total promos in file: {len(promos)}")

Promo features added.
  Train active promo days: 1707
  Test  active promo days: 0
  Total promos in file: 50


## Cell 7: Step 5 — Web Traffic Features

In [7]:
web = pd.read_csv(f"{RAW}/web_traffic.csv", parse_dates=["date"])

# Aggregate by date (multiple traffic sources per day -> sum/mean)
web_daily = web.groupby("date").agg(
    sessions            = ("sessions",                "sum"),
    unique_visitors     = ("unique_visitors",          "sum"),
    page_views          = ("page_views",               "sum"),
    bounce_rate         = ("bounce_rate",              "mean"),
    avg_session_dur     = ("avg_session_duration_sec", "mean"),
).reset_index().rename(columns={"date": "Date"})

print(f"Web traffic daily: {web_daily.shape} | {web_daily['Date'].min().date()} -> {web_daily['Date'].max().date()}")

# Web traffic only covers training period; for test use lag-365 proxy
web_lag365 = web_daily.copy()
web_lag365["Date"] = web_lag365["Date"] + pd.DateOffset(days=365)
rename_map = {c: f"{c}_lag365" for c in web_daily.columns if c != "Date"}
web_lag365 = web_lag365.rename(columns=rename_map)

# Merge: train gets direct traffic, test gets lag365 proxy
train_feat = train_feat.merge(web_daily,  on="Date", how="left")
test_feat  = test_feat.merge(web_lag365,  on="Date", how="left")

# Standardize column names in test
web_cols = ["sessions", "unique_visitors", "page_views", "bounce_rate", "avg_session_dur"]
for col in web_cols:
    lag_col = f"{col}_lag365"
    if lag_col in test_feat.columns:
        test_feat[col] = test_feat[lag_col]
        test_feat.drop(columns=[lag_col], inplace=True)

# Fill missing values with training median
for col in web_cols:
    median_val = train_feat[col].median()
    train_feat[col] = train_feat[col].fillna(median_val)
    test_feat[col]  = test_feat[col].fillna(median_val)

print("Web traffic features added.")
print(f"  NaN in sessions (train): {train_feat['sessions'].isna().sum()}")
print(f"  NaN in sessions (test):  {test_feat['sessions'].isna().sum()}")

Web traffic daily: (3652, 6) | 2013-01-01 -> 2022-12-31
Web traffic features added.
  NaN in sessions (train): 0
  NaN in sessions (test):  0


## Cell 8: Step 6 — Inventory Features (monthly aggregation)

In [8]:
inventory = pd.read_csv(f"{RAW}/inventory.csv")
print(f"Inventory: {inventory.shape} | columns: {list(inventory.columns)}")

# Monthly aggregation
inv_monthly = inventory.groupby(["year", "month"]).agg(
    avg_fill_rate  = ("fill_rate",    "mean"),
    stockout_rate  = ("stockout_flag","mean"),
    avg_stock      = ("stock_on_hand","mean"),
).reset_index()

# Train: direct merge by year + month
train_feat = train_feat.merge(inv_monthly, on=["year", "month"], how="left")

# Test: use prior-year inventory as proxy (year+1 in lag, merging to test year)
inv_lag = inv_monthly.copy()
inv_lag["year"] = inv_lag["year"] + 1
inv_lag = inv_lag.rename(columns={
    "avg_fill_rate": "avg_fill_rate_lag1y",
    "stockout_rate": "stockout_rate_lag1y",
    "avg_stock":     "avg_stock_lag1y",
})
test_feat = test_feat.merge(inv_lag, on=["year", "month"], how="left")

# Standardize column names in test
for col in ["avg_fill_rate", "stockout_rate", "avg_stock"]:
    lag_col = f"{col}_lag1y"
    if lag_col in test_feat.columns:
        test_feat[col] = test_feat[lag_col]
        test_feat.drop(columns=[lag_col], inplace=True)

# Fill NaN with training medians (with safe fallbacks)
inv_cols = ["avg_fill_rate", "stockout_rate", "avg_stock"]
fallback  = {"avg_fill_rate": 0.96, "stockout_rate": 0.05, "avg_stock": 500.0}
for col in inv_cols:
    median_val = train_feat[col].median() if col in train_feat.columns else fallback[col]
    if col in train_feat.columns:
        train_feat[col] = train_feat[col].fillna(median_val)
    if col in test_feat.columns:
        test_feat[col] = test_feat[col].fillna(median_val)
    else:
        test_feat[col] = median_val

print("Inventory features added.")
print(f"  NaN avg_fill_rate (train): {train_feat['avg_fill_rate'].isna().sum()}")
print(f"  NaN avg_fill_rate (test):  {test_feat['avg_fill_rate'].isna().sum()}")

Inventory: (60247, 17) | columns: ['snapshot_date', 'product_id', 'stock_on_hand', 'units_received', 'units_sold', 'stockout_days', 'days_of_supply', 'fill_rate', 'stockout_flag', 'overstock_flag', 'reorder_flag', 'sell_through_rate', 'product_name', 'category', 'segment', 'year', 'month']
Inventory features added.
  NaN avg_fill_rate (train): 0
  NaN avg_fill_rate (test):  0


## Cell 9: Step 7 — COGS Ratio Feature

In [9]:
# COGS ratio as additional feature (stable ~0.8746 per EDA)
train_feat["cogs_ratio"] = np.where(
    train_feat["Revenue"] > 0,
    train_feat["COGS"] / train_feat["Revenue"],
    np.nan
)

# Lag-365 of COGS ratio using lookup — safe for test dates
cogs_ratio_series  = sales.copy()
cogs_ratio_series["cogs_ratio"] = cogs_ratio_series["COGS"] / cogs_ratio_series["Revenue"]
cogs_ratio_lookup  = cogs_ratio_series.set_index("Date")["cogs_ratio"]

for df in [train_feat, test_feat]:
    df["cogs_ratio_lag365"] = safe_lag(df, cogs_ratio_lookup, 365)
    df["cogs_ratio_lag365"] = df["cogs_ratio_lag365"].fillna(0.8746)

print(f"COGS ratio feature — train mean: {train_feat['cogs_ratio'].mean():.4f}")
print(f"COGS ratio lag365  — train mean: {train_feat['cogs_ratio_lag365'].mean():.4f}")
print(f"COGS ratio lag365  — test  mean: {test_feat['cogs_ratio_lag365'].mean():.4f}")

COGS ratio feature — train mean: 0.8746
COGS ratio lag365  — train mean: 0.8742
COGS ratio lag365  — test  mean: 0.8776


## Cell 10: Step 8 — Year-Level Trend Feature

In [10]:
# Annual mean Revenue — captures trend level
annual_mean = sales.groupby(sales["Date"].dt.year)["Revenue"].mean()
print("Annual mean Revenue:")
print(annual_mean.to_string())

train_feat["year_mean_rev"] = train_feat["year"].map(annual_mean)

# For test: project from 2022 using last YoY growth factor
mean_2021  = annual_mean.get(2021, annual_mean.iloc[-2])
mean_2022  = annual_mean.get(2022, annual_mean.iloc[-1])
yoy_factor = mean_2022 / mean_2021

test_feat["year_mean_rev"] = test_feat["year"].map({
    2023: mean_2022 * yoy_factor,
    2024: mean_2022 * (yoy_factor ** 2),
})
test_feat["year_mean_rev"] = test_feat["year_mean_rev"].fillna(mean_2022)

print(f"\nYoY growth factor (2021->2022): {yoy_factor:.4f}")
print(f"Projected 2023 annual mean: {mean_2022 * yoy_factor:,.0f}")
print(f"Projected 2024 annual mean: {mean_2022 * yoy_factor**2:,.0f}")

Annual mean Revenue:
Date
2012    4.096673e+06
2013    4.540190e+06
2014    5.128345e+06
2015    5.177901e+06
2016    5.750384e+06
2017    5.236067e+06
2018    5.068829e+06
2019    3.114524e+06
2020    2.881181e+06
2021    2.857643e+06
2022    3.204791e+06

YoY growth factor (2021->2022): 1.1215
Projected 2023 annual mean: 3,594,111
Projected 2024 annual mean: 4,030,725


## Cell 11: Step 9 — Final Cleanup & Save

In [11]:
# Define final feature columns (ordered by category)
FEATURE_COLS = [
    # --- Temporal ---
    "year", "month", "quarter", "day_of_month", "day_of_week",
    "day_of_year", "week_of_year", "is_weekend", "is_month_start",
    "is_month_end", "is_quarter_start", "is_quarter_end",
    "month_sin", "month_cos", "dow_sin", "dow_cos", "doy_sin", "doy_cos",
    # --- Lag features (no leakage) ---
    "rev_lag_365", "cogs_lag_365", "rev_lag_730", "cogs_lag_730",
    "rev_lag_364", "rev_lag_366",
    "rev_rolling7_lag365", "rev_rolling30_lag365",
    "yoy_ratio",
    # --- Holiday ---
    "is_holiday", "is_tet", "is_national_day",
    "days_to_holiday", "days_since_holiday",
    "is_near_holiday", "is_post_holiday",
    # --- Promotion ---
    "promo_count", "avg_discount", "max_discount", "has_promo",
    # --- Web traffic ---
    "sessions", "unique_visitors", "page_views", "bounce_rate", "avg_session_dur",
    # --- Inventory ---
    "avg_fill_rate", "stockout_rate", "avg_stock",
    # --- COGS ratio ---
    "cogs_ratio_lag365",
    # --- Trend ---
    "year_mean_rev",
]

# Only keep columns that exist in BOTH dataframes
FEATURE_COLS = [c for c in FEATURE_COLS if c in train_feat.columns and c in test_feat.columns]
print(f"Final feature count: {len(FEATURE_COLS)}")
print("Features:", FEATURE_COLS)

# Check NaN in test features before fillna
print("\nNaN counts in test features (before fillna):")
nan_found = False
for col in FEATURE_COLS:
    n = test_feat[col].isna().sum()
    if n > 0:
        print(f"  {col}: {n} NaNs")
        nan_found = True
if not nan_found:
    print("  None — all clean!")

# Final fillna with 0 (LightGBM handles NaN natively but good hygiene)
for col in FEATURE_COLS:
    train_feat[col] = train_feat[col].fillna(0)
    test_feat[col]  = test_feat[col].fillna(0)

# Verify no remaining NaN
assert test_feat[FEATURE_COLS].isna().sum().sum() == 0, "NaN found in test features after fillna!"
assert train_feat[FEATURE_COLS].isna().sum().sum() == 0, "NaN found in train features after fillna!"
print("\nNaN check passed: no NaN in feature columns.")

# Save to parquet
train_save = train_feat[["Date", "Revenue", "COGS"] + FEATURE_COLS]
test_save  = test_feat[["Date"] + FEATURE_COLS]

train_save.to_parquet(f"{PROCESSED}/train_features.parquet", index=False)
test_save.to_parquet(f"{PROCESSED}/test_features.parquet",  index=False)

print(f"\nSaved train_features.parquet: {train_save.shape}")
print(f"Saved test_features.parquet:  {test_save.shape}")

# Save feature list for model notebooks
with open(f"{PROCESSED}/feature_cols.json", "w") as f:
    json.dump(FEATURE_COLS, f, indent=2)
print(f"Saved feature_cols.json: {len(FEATURE_COLS)} features")

Final feature count: 48
Features: ['year', 'month', 'quarter', 'day_of_month', 'day_of_week', 'day_of_year', 'week_of_year', 'is_weekend', 'is_month_start', 'is_month_end', 'is_quarter_start', 'is_quarter_end', 'month_sin', 'month_cos', 'dow_sin', 'dow_cos', 'doy_sin', 'doy_cos', 'rev_lag_365', 'cogs_lag_365', 'rev_lag_730', 'cogs_lag_730', 'rev_lag_364', 'rev_lag_366', 'rev_rolling7_lag365', 'rev_rolling30_lag365', 'yoy_ratio', 'is_holiday', 'is_tet', 'is_national_day', 'days_to_holiday', 'days_since_holiday', 'is_near_holiday', 'is_post_holiday', 'promo_count', 'avg_discount', 'max_discount', 'has_promo', 'sessions', 'unique_visitors', 'page_views', 'bounce_rate', 'avg_session_dur', 'avg_fill_rate', 'stockout_rate', 'avg_stock', 'cogs_ratio_lag365', 'year_mean_rev']

NaN counts in test features (before fillna):
  rev_lag_365: 183 NaNs
  cogs_lag_365: 183 NaNs
  rev_lag_364: 184 NaNs
  rev_lag_366: 182 NaNs
  rev_rolling7_lag365: 180 NaNs
  rev_rolling30_lag365: 168 NaNs

NaN check pa

## Cell 12: Step 10 — Visualization & Validation

In [12]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# 1. rev_lag_365 vs actual Revenue (should correlate strongly)
ax = axes[0, 0]
sample = train_feat[train_feat["year"] == 2022]
ax.scatter(sample["rev_lag_365"], sample["Revenue"], alpha=0.3, s=5, color="steelblue")
ax.set_xlabel("Revenue lag_365")
ax.set_ylabel("Actual Revenue")
ax.set_title("Lag_365 vs Actual Revenue (2022 validation)")
corr = sample[["rev_lag_365", "Revenue"]].corr().iloc[0, 1]
ax.text(0.05, 0.95, f"r = {corr:.3f}", transform=ax.transAxes,
        fontsize=12, va="top", color="darkred")

# 2. Revenue vs Lag_365 distribution overlay
ax = axes[0, 1]
ax.hist(train_feat["rev_lag_365"].dropna() / 1e6, bins=50, alpha=0.6, label="lag_365", color="orange")
ax.hist(train_feat["Revenue"] / 1e6,              bins=50, alpha=0.6, label="Actual",  color="steelblue")
ax.set_xlabel("Revenue (M VND)")
ax.set_title("Revenue vs Lag_365 Distribution")
ax.legend()

# 3. Monthly avg of rev_lag_365 on test set
ax = axes[1, 0]
test_monthly = test_feat.groupby("month")["rev_lag_365"].mean() / 1e6
ax.bar(test_monthly.index, test_monthly.values, color="coral")
ax.set_title("Test Set: avg rev_lag_365 by Month")
ax.set_xlabel("Month")
ax.set_ylabel("Revenue lag_365 (M VND)")
ax.set_xticks(range(1, 13))
ax.set_xticklabels(["Jan","Feb","Mar","Apr","May","Jun",
                     "Jul","Aug","Sep","Oct","Nov","Dec"], fontsize=8)

# 4. Feature correlation heatmap
ax = axes[1, 1]
key_features = ["rev_lag_365", "rev_lag_730", "rev_rolling7_lag365",
                "is_holiday", "promo_count", "sessions", "year_mean_rev"]
key_features = [c for c in key_features if c in train_feat.columns]
corr_matrix  = train_feat[key_features + ["Revenue"]].corr()
im = ax.imshow(corr_matrix, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr_matrix.columns)))
ax.set_yticks(range(len(corr_matrix.columns)))
ax.set_xticklabels(corr_matrix.columns, rotation=45, ha="right", fontsize=8)
ax.set_yticklabels(corr_matrix.columns, fontsize=8)
ax.set_title("Feature Correlation Matrix (incl. Revenue)")
plt.colorbar(im, ax=ax)

# Annotate cells
for i in range(len(corr_matrix)):
    for j in range(len(corr_matrix.columns)):
        val = corr_matrix.iloc[i, j]
        ax.text(j, i, f"{val:.2f}", ha="center", va="center",
                fontsize=6, color="white" if abs(val) > 0.5 else "black")

plt.suptitle("Feature Engineering Validation", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("../report/figures/06_features.png", dpi=120, bbox_inches="tight")
plt.show()

print(f"\nCorrelation rev_lag_365 <-> Revenue (full train): "
      f"{train_feat[['rev_lag_365','Revenue']].corr().iloc[0,1]:.4f}")
print("Feature validation figure saved to ../report/figures/06_features.png")


Correlation rev_lag_365 <-> Revenue (full train): 0.6555
Feature validation figure saved to ../report/figures/06_features.png
